%md
# Fetch Access Token

This task extracts and persists a task value `access_token` for the job, which is used downstream to make query with GraphQL.

1. Import required dependencies.
2. Check expiry of currently stored token.
3. Read and build new credentials from secrets manager if needed.
4. Set downstream task value.
5. Run task.

In [0]:
import base64
import json
import time
import httpx
from typing import Any

from databricks.sdk import WorkspaceClient
import pandas as pd
from pydantic import (
    BaseModel,
    Field,
    field_validator,
)

In [0]:
from settings import fflogs_settings as settings

In [0]:
def is_token_expired(
    token: str, 
    buffer_seconds: int = 60
) -> bool:
    """Returns True if the token is expired or about to expire."""
    payload = token.split(".")[1]
    # base64url needs padding to be a multiple of 4
    padded = payload + "=" * (4 - len(payload) % 4)
    claims = json.loads(base64.urlsafe_b64decode(padded))
    
    return claims["exp"] < time.time() + buffer_seconds

In [0]:
async def get_new_access_token(
    client: httpx.AsyncClient, 
    username: str,
    password: str, 
) -> str:
    """Fetches a client-credentials OAuth2 access token using httpx.AsyncClient.

    Returns:
        The access token string.
    """
    _FFLOGS_TOKEN_URL = "https://www.fflogs.com/oauth/token"

    response = await client.post(
        _FFLOGS_TOKEN_URL,
        auth=(username, password),
        data={"grant_type": "client_credentials"},
    )

    response.raise_for_status()

    return response.json()["access_token"]


In [0]:
def create_secret(
    workspace: WorkspaceClient,
    scope: str,
    key: str,
    value: str,
) -> None:
    workspace.secrets.put_secret(
        scope=scope,
        key=key,
        stringValue=value
    )

    print(f"✅ Secret {key} added successfully to {scope}!")

In [0]:
async def refresh_access_token_if_expired() -> None:
    token = settings.fflogs_access_token
    token_expired = is_token_expired(token)
    if token_expired:
        async with httpx.AsyncClient() as client:
            new_token = await get_new_access_token(
                client,
                settings.fflogs_client_id,
                settings.fflogs_client_secret
            )
        create_secret(
            workspace=WorkspaceClient(),
            scope=settings.SECRET_SCOPE,
            key="fflogs_access_token",
            stringValue=new_token
        )
        print(f"Token refreshed and committed to secrets --> {new_token}")
    else:
        print(f"Token is still valid --> {token}")


In [0]:
await refresh_access_token_if_expired()